Estos son todos los imports necesarios para poder ejecutar todas las celdas de este archivo.

In [14]:
import cv2
import csv
import math 
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import json
import random
import shutil
from pathlib import Path
import torch
import easyocr

In [ ]:
#Se carga el modelo
model = YOLO('yolo11n.pt')

#Etiquetas de las distintas clases que el modelo va a detectar
classNames = ["person", "bicycle", "car"]

#Se deine el video que se va a analizar
vid = cv2.VideoCapture("C0142.MP4")
#Se crea un diccionario para almacenar los IDs de los objetos detectados
track_history = defaultdict(lambda: [])

#Para guardar el vídeo procesado se necesita el códec, los fps y el tamaño del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

#Se guarda el vídeo procesado
out = cv2.VideoWriter("C0142_procesado_blur.mp4", fourcc, fps, (width, height))

#Se inicializan los contadores y los sets para almacenar los IDs contados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

while(True):      
    #Se extrae cada fotograma del video
    ret, img = vid.read()
  
    #si la imagen es válida
    if ret:  
        #Se realiza seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])
        
        #Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                #Se definen las cajas que rodean al objeto detectado
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                #De acuerdo con uno de los apartados extras de la práctica,
                #se aplica un blur a los objetos detectados para anonimizarlos
                roi = img[y1:y2, x1:x2]
                roi_blur = cv2.GaussianBlur(roi, (51, 51), 0)
                img[y1:y2, x1:x2] = roi_blur

                #Se establece la etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                #Se establece la confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza =",confidence)

                #Se establece la clase
                cls = int(box.cls[0])
                print("Clase =", classNames[cls])

       #Actualiza los contadores si el ID de seguimiento no ha sido contado previamente
                if box.id is not None:
                    nombre_clase = classNames[cls]
                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                #Se convierte el identificador numérico de clase a un color RGB
                if cls == 0:
                    R = 128
                    G = 0
                    B = 0
                elif cls == 1:
                    R = 0
                    G = 128
                    B = 0
                elif cls == 2:
                    R = 0
                    G = 0
                    B = 128

                #Se dibuja el contenedor en función de la clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

       #Se dibujan los contadores en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

       #Se guarda el fotograma procesado en el vídeo de salida
        out.write(img)

        #Se muestra el fotograma
        cv2.imshow('Vid', img)
    else:
        break
    
    #Se detiene la ejecución pulsando ESC
    if cv2.waitKey(20) == 27:
        break
  
#Se libera el objeto de captura
vid.release()
#Se libera el objeto de escritura de vídeo
out.release()
#Se cierra la ventana emergente
cv2.destroyAllWindows()


In [ ]:
#Se carga del modelo YOLO y se definen de las clases que se van a detectar
model = YOLO("yolo11n.pt")

classNames = ["person", "bicycle", "car"]

cap = cv2.VideoCapture("C0142.MP4")

#Se obtienen las características del vídeo original
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

#Se crea el vídeo de salida con las mismas dimensiones y FPS que el original
out = cv2.VideoWriter("C0142_direccion.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

#Se define la línea de referencia para contar los vehículos
#Las coordenadas se calculan proporcionalmente al tamaño del vídeo
linea_vehiculos_y = int(height * 430 / 928)
linea_vehiculos_x1 = int(width * 500 / 1650)
linea_vehiculos_x2 = width - 5

#Se define el segmento de referencia para contar los peatones
#Se sitúa sobre la acera y sigue aproximadamente su dirección
persona_x1 = int(width * 450 / 1650)
persona_y1 = int(height * 325 / 928)

persona_x2 = int(width * 500 / 1650)
persona_y2 = int(height * 390 / 928)

#Se establece un margen alrededor de la línea de vehículos para evitar
#contar cruces provocados por pequeñas oscilaciones de la detección
margen = 12

#Contadores de objetos según su dirección de movimiento
flujo = {"person": {"acercandose": 0, "alejandose": 0}, "bicycle": {"acercandose": 0, "alejandose": 0}, 
         "car": {"acercandose": 0, "alejandose": 0}}

#Se crean los siguientes diccionarios para almacenar la última posición conocida de cada objeto
posiciones = {}
posiciones_personas = {}

#Contados respresenta el conjunto de IDs que ya han sido contabilizados
contados = set()

while cap.isOpened():
    ret, img = cap.read()

    if not ret:
        break

    #Se detecta y realiza el seguimiento de personas, bicicletas y coches
    results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

    for box in results[0].boxes:

        #Solo se procesan detecciones que tengan un ID de seguimiento
        if box.id is None:
            continue

        #Se obbtiene las coordenadas de la caja, el ID y la clase del objeto
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        track_id = int(box.id[0])
        cls = int(box.cls[0])
        nombre = classNames[cls]

        #Se utiliza el centro inferior de la caja como referencia
        #para determinar la posición del objeto respecto a las líneas
        cx = (x1 + x2) // 2
        cy = y2

        #Se analiza la dirección de coches y bicicletas
        if nombre in ["car", "bicycle"]:

            #El objeto se encuentra dentro del tramo horizontal
            #donde está definida la línea de referencia
            if linea_vehiculos_x1 <= cx <= linea_vehiculos_x2:

                #Se determina en qué lado de la línea se encuentra
                #El margen permite ignorar posiciones demasiado próximas
                if cy < linea_vehiculos_y - margen:
                    lado_actual = -1
                elif cy > linea_vehiculos_y + margen:
                    lado_actual = 1
                else:
                    lado_actual = 0

                #Se guarda la primera posición válida del objeto
                if track_id not in posiciones:
                    if lado_actual != 0:
                        posiciones[track_id] = lado_actual

                #Si el objeto todavía no se ha contado, comprobar
                #si ha cambiado de lado respecto a su posición anterior
                elif track_id not in contados:

                    lado_anterior = posiciones[track_id]

                    if lado_actual != 0:

                        #Un cambio de arriba hacia abajo indica que se acerca
                        if lado_anterior == -1 and lado_actual == 1:
                            flujo[nombre]["acercandose"] += 1
                            contados.add(track_id)

                        #Un cambio de abajo hacia arriba indica que se aleja
                        elif lado_anterior == 1 and lado_actual == -1:
                            flujo[nombre]["alejandose"] += 1
                            contados.add(track_id)

                        #Se actualiza el último lado conocido del objeto
                        posiciones[track_id] = lado_actual

        #Se analiza la dirección de los peatones
        elif nombre == "person":

            #Se guarda la posición inicial del peatón
            if track_id not in posiciones_personas:
                posiciones_personas[track_id] = (cx, cy)

            elif track_id not in contados:

                #Se recupera la posición anterior para compararla con la actual
                anterior_x, anterior_y = posiciones_personas[track_id]

                #Se calcula el lado de la línea en el que se encuentra
                #cada posición mediante el producto vectorial
                lado_anterior = (
                (persona_x2 - persona_x1) * (anterior_y - persona_y1) - (persona_y2 - persona_y1) * (anterior_x - persona_x1))

                lado_actual = (
                (persona_x2 - persona_x1) * (cy - persona_y1) - (persona_y2 - persona_y1) * (cx - persona_x1))

                #Si los resultados tienen signos opuestos,
                #el peatón ha pasado de un lado de la línea al otro
                if lado_anterior * lado_actual < 0:

                    #Se comprueba que el cruce se ha producido cerca
                    #del segmento definido para los peatones
                    medio_x = (anterior_x + cx) // 2
                    medio_y = (anterior_y + cy) // 2

                    cerca_linea = (
                        min(persona_x1, persona_x2) - 30 <= medio_x <= max(persona_x1, persona_x2) + 30
                        and min(persona_y1, persona_y2) - 30 <= medio_y <= max(persona_y1, persona_y2) + 30
                    )

                    if cerca_linea:
                        #Se clasifica el sentido del movimiento según
                        #el lado de la línea al que se ha desplazado
                        if lado_anterior > 0 and lado_actual < 0:
                            flujo["person"]["alejandose"] += 1

                        else:
                            flujo["person"]["acercandose"] += 1

                        contados.add(track_id)

                #Se actualiza la posición del peatón para el siguiente fotograma
                posiciones_personas[track_id] = (cx, cy)

        #Se dibuja la caja delimitadora y el identificador del objeto
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(img, f"{nombre} ID:{track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    #Se dibuja la línea roja de referencia para coches y bicicletas
    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y), (linea_vehiculos_x2, linea_vehiculos_y), (0, 0, 255), 3)

    #Se dibujan los límites del margen de seguridad
    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y - margen), (linea_vehiculos_x2, linea_vehiculos_y - margen), (0, 255, 255), 1)

    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y + margen), (linea_vehiculos_x2, linea_vehiculos_y + margen), (0, 255, 255), 1)

    #Se dibuja la línea azul utilizada para analizar el movimiento de los peatones
    cv2.line(img, (persona_x1, persona_y1), (persona_x2, persona_y2), (255, 0, 0), 3)

    #Se muestran los contadores de cada clase y dirección en el fotograma
    y = 30

    for nombre in classNames:
        texto = (f"{nombre}: "f"acercandose {flujo[nombre]['acercandose']} "f"alejandose {flujo[nombre]['alejandose']}")

        cv2.putText(img, texto, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        y += 25

    #Se guarda y muestra el fotograma procesado
    out.write(img)
    cv2.imshow("Analisis de direccion", img)

    #Se detiene la ejecución si se pulsa ESC
    if cv2.waitKey(20) == 27:
        break

#Se liberan los recursos y cierran las ventanas
cap.release()
out.release()
cv2.destroyAllWindows()

#Se muestarn los resultados finales del análisis de dirección
print("Resultados del flujo:")

for nombre in classNames:
    print(nombre, flujo[nombre])

In [ ]:
#Se localiza el directorio del dataset original
raiz_dataset = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\matriculas")

#Se definen las rutas de las imágenes, anotaciones y directorio de salida
imagenes_dir = raiz_dataset / "images"
anotaciones_dir = raiz_dataset / "annotations"
salida_dir = raiz_dataset / "dataset_yolo"

#Se buscan las imágenes y los archivos JSON disponibles
extensiones = [".jpg", ".jpeg"]

imagenes = {}
for ext in extensiones:
    for archivo in imagenes_dir.glob("*" + ext):
        imagenes[archivo.stem] = archivo

anotaciones = {archivo.stem: archivo for archivo in anotaciones_dir.glob("*.json")}

#Se comprueba que imágenes tienen un archivo JSON asociado
comunes = sorted(set(imagenes) & set(anotaciones))

print("\nImágenes encontradas:", len(imagenes))
print("JSON encontrados:", len(anotaciones))
print("Parejas encontradas:", len(comunes))
print("Imágenes sin JSON:", len(set(imagenes) - set(anotaciones)))
print("JSON sin imagen:", len(set(anotaciones) - set(imagenes)))

#Se preparan las anotaciones válidas para el formato YOLO
datos = []
json_invalidos = 0
anotaciones_descartadas = 0

for nombre in comunes:
    try:
        with open(anotaciones[nombre], "r", encoding="utf-8") as f:
            contenido = json.load(f)

        #Se comprueba que el contenido del JSON tenga el formato esperado
        if not isinstance(contenido, list):
            json_invalidos += 1
            continue

        etiquetas = []

        for objeto in contenido:
            if not isinstance(objeto, dict):
                anotaciones_descartadas += 1
                continue

            #Se obtienen las coordenadas del rectángulo de la matrícula
            rect = objeto.get("rectangle")

            if not isinstance(rect, list) or len(rect) != 4:
                anotaciones_descartadas += 1
                continue

            #Se convierten las coordenadas a valores numéricos
            try:
                x, y, w, h = map(float, rect)
            except (TypeError, ValueError):
                anotaciones_descartadas += 1
                continue

            #Se descartan las anotaciones con valores fuera del rango permitido o con dimensiones nulas o negativas
            if not all(0 <= v <= 1 for v in [x, y, w, h]) or w <= 0 or h <= 0:
                anotaciones_descartadas += 1
                continue

            #Se convierte cada anotación al formato de YOLO
            etiquetas.append(f"0 {x:.10f} {y:.10f} {w:.10f} {h:.10f}")

        if etiquetas:
            datos.append((imagenes[nombre], etiquetas))

    except (json.JSONDecodeError, OSError, TypeError):
        json_invalidos += 1

#Se mezclan los datos y se dividen en entrenamiento, validación y prueba
random.seed(1)
random.shuffle(datos)

total = len(datos)
n_train = int(total * 0.70)
n_val = int(total * 0.20)

conjuntos = {"train": datos[:n_train], "val": datos[n_train:n_train + n_val], "test": datos[n_train + n_val:]}

#Se crea la estructura de carpetas para el nuevo dataset
if salida_dir.exists():
    shutil.rmtree(salida_dir)

for conjunto, elementos in conjuntos.items():
    (salida_dir / "images" / conjunto).mkdir(parents=True, exist_ok=True)
    (salida_dir / "labels" / conjunto).mkdir(parents=True, exist_ok=True)

    #Se copian las imágenes y se guardan sus correspondientes archivos de etiquetas
    for imagen, etiquetas in elementos:
        shutil.copy2(imagen, salida_dir / "images" / conjunto / imagen.name)

        etiqueta_file = salida_dir / "labels" / conjunto / (imagen.stem + ".txt")

        with open(etiqueta_file, "w", encoding="utf-8") as f:
            f.write("\n".join(etiquetas))

#Se crea el archivo de configuración necesario para entrenar YOLO
yaml = f"""path: {salida_dir.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['matricula']
"""

with open(salida_dir / "data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml)

#Se muestra un resumen de la preparación del dataset
print("\nDataset preparado correctamente")
print("Imágenes válidas:", total)
print("JSON inválidos:", json_invalidos)
print("Anotaciones descartadas:", anotaciones_descartadas)

for conjunto, elementos in conjuntos.items():
    print(f"{conjunto}: {len(elementos)} imágenes")

print("\nDataset guardado en:", salida_dir.resolve())

In [ ]:
#Se definen las rutas del dataset y del modelo preentrenado
dataset_dir = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\matriculas\dataset_yolo")
modelo_path = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\yolo11n.pt")

#Se selecciona el dispositivo disponible para realizar el entrenamiento
device = 0 if torch.cuda.is_available() else "cpu"

print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

#Se carga el modelo YOLO preentrenado
model = YOLO(modelo_path)

#Se entrena el detector de matrículas utilizando el dataset preparado
resultados = model.train(
    data=str(dataset_dir / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=8,
    device=device,
    workers=0,
    project="../matriculas/runs",
    name="train_matriculas",
    exist_ok=True,
    plots=True
)

In [ ]:
#Se localiza la ruta del mejor modelo obtenido durante el entrenamiento
modelo_path = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt")

#Se localiza el directorio del dataset preparado
dataset_dir = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\matriculas\dataset_yolo")

#Se carga el mejor modelo entrenado
model = YOLO(str(modelo_path))

#Se evalúa el rendimiento del modelo utilizando las imágenes del conjunto de prueba
resultados_test = model.val(
    data=str(dataset_dir / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project=str(dataset_dir.parent / "runs"),
    name="test_matriculas"
)

In [ ]:
#Se define la ruta del vídeo que se va a procesar
video_path = Path(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\C0142.MP4")

#Se carga el modelo YOLO entrenado para detectar matrículas
model = YOLO(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt")

#Se abre el vídeo y se comprueba que se haya cargado correctamente
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir el video")

#Se obtienen las características del vídeo original
fps = cap.get(cv2.CAP_PROP_FPS)
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#Se crea el vídeo de salida conservando las características del original
salida_path = video_path.parent / (video_path.stem + "_matriculas.mp4")

writer = cv2.VideoWriter(str(salida_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

#Se inicializan los contadores para registrar el procesamiento del vídeo
total_detecciones = 0
fotograma = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    #Se detectan las matrículas presentes en el fotograma
    resultados = model.predict(frame, imgsz=640, conf=0.25, verbose=False)

    #Se dibujan las detecciones y se registra su número total
    for box in resultados[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        confianza = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"Matricula {confianza:.2f}", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        total_detecciones += 1

    #Se guarda el fotograma procesado en el vídeo de salida
    writer.write(frame)
    fotograma += 1

    #Se muestra el progreso del procesamiento cada 100 fotogramas
    if fotograma % 100 == 0:
        print(f"Procesados: {fotograma}/{total_frames}")

#Se liberan los recursos utilizados para procesar el vídeo
cap.release()
writer.release()

#Se muestra un resumen del procesamiento realizado
print("\nProcesamiento terminado")
print("Fotogramas:", fotograma)
print("Detecciones totales:", total_detecciones)
print("Video generado:", salida_path.resolve())

In [ ]:
#Se cargan los modelos para detectar objetos y matrículas
model = YOLO('yolo11n.pt')  # Personas, bicicletas y coches
model_matriculas = YOLO(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt")

#Se definen las etiquetas correspondientes a las distintas clases detectadas
classNames = ["person", "bicycle", "car"]

#Se inicia la captura del vídeo
vid = cv2.VideoCapture("C0142.MP4")

track_history = defaultdict(lambda: [])

#Se obtiene la información del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

#Se prepara el archivo de vídeo donde se guardarán los fotogramas procesados
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

#Se crea el archivo CSV para almacenar los resultados de las detecciones
archivo_csv = open("C0142_resultados.csv", "w", newline="", encoding="utf-8")
campos = [
    "fotograma", "tipo_objeto", "confianza",
    "identificador_tracking", "x1", "y1", "x2", "y2",
    "matricula_en_su_caso", "confianza_matricula",
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

escritor = csv.DictWriter(archivo_csv, fieldnames=campos)
escritor.writeheader()

#Se inicializan los contadores y los conjuntos que almacenan los identificadores ya contabilizados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

#Se inicializa el contador de fotogramas
num_fotograma = 0

while(True):

    #Se obtiene el siguiente fotograma del vídeo
    ret, img = vid.read()

    #Se comprueba si se ha obtenido una imagen válida
    if ret:

        num_fotograma += 1

        #Se realiza el seguimiento de personas, bicicletas y coches
        results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

        #Se detectan las matrículas presentes en el fotograma
        results_matriculas = model_matriculas.predict(img, conf=0.25, verbose=False)

        #Se prepara una lista para almacenar las matrículas detectadas
        matriculas = []

        for r_mat in results_matriculas:
            for box_mat in r_mat.boxes:

                #Se obtienen las coordenadas de cada matrícula
                mx1, my1, mx2, my2 = box_mat.xyxy[0].cpu().numpy()
                mx1, my1, mx2, my2 = int(mx1), int(my1), int(mx2), int(my2)

                #Se ajustan las coordenadas para evitar que queden fuera de la imagen
                mx1 = max(0, mx1)
                my1 = max(0, my1)
                mx2 = min(width, mx2)
                my2 = min(height, my2)

                #Se obtiene la confianza de la detección de la matrícula
                confianza_mat = float(box_mat.conf[0])

                #Se almacena la información de la matrícula detectada
                matriculas.append({
                    "x1": mx1,
                    "y1": my1,
                    "x2": mx2,
                    "y2": my2,
                    "confianza": confianza_mat,
                    "confianza_ocr": "None",
                    "texto": "None"
                })

        #Se recorren las detecciones de personas, bicicletas y coches
        for r in results:

            boxes = r.boxes

            for box in boxes:

                #Se obtienen las coordenadas del objeto detectado
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                #Se ajustan las coordenadas para evitar que queden fuera de la imagen
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                #Se obtiene la confianza de la detección
                confidence = float(box.conf[0])
                print("Confianza =", round(confidence, 2))

                #Se identifica la clase del objeto detectado
                cls = int(box.cls[0])
                nombre_clase = classNames[cls]
                print("Clase =", nombre_clase)

                #Se obtiene el identificador asignado al objeto durante el seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0]))
                else:
                    track_id = ""

                #Se inicializa la información de la matrícula asociada al objeto
                matricula_asociada = None

                #Se buscan matrículas asociadas únicamente a las detecciones de coches
                if cls == 2:

                    for mat in matriculas:

                        #Se calcula el centro de la matrícula detectada
                        centro_x = (mat["x1"] + mat["x2"]) // 2
                        centro_y = (mat["y1"] + mat["y2"]) // 2

                        #Se comprueba si el centro de la matrícula se encuentra dentro del coche
                        if (x1 <= centro_x <= x2 and
                            y1 <= centro_y <= y2):

                            #Se selecciona la matrícula con mayor confianza si hay varias asociadas
                            if (matricula_asociada is None or mat["confianza"] > matricula_asociada["confianza"]):
                                matricula_asociada = mat

                #Se actualizan los contadores cuando aparece un identificador que todavía no se ha contabilizado
                if box.id is not None:

                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                #Se prepara la información de la matrícula para almacenarla en el CSV
                if matricula_asociada is not None:

                    tiene_matricula = "Si"
                    confianza_matricula = matricula_asociada["confianza"]
                    mx1 = matricula_asociada["x1"]
                    my1 = matricula_asociada["y1"]
                    mx2 = matricula_asociada["x2"]
                    my2 = matricula_asociada["y2"]
                    texto_matricula = matricula_asociada["texto"]

                else:
                    tiene_matricula = "No"
                    confianza_matricula = ""
                    mx1 = ""
                    my1 = ""
                    mx2 = ""
                    my2 = ""
                    texto_matricula = ""

                #Se registra en el CSV la información del objeto y, si corresponde, de su matrícula
                escritor.writerow({
                    "fotograma": num_fotograma,
                    "tipo_objeto": nombre_clase,
                    "confianza": confidence,
                    "identificador_tracking": track_id,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "matricula_en_su_caso": tiene_matricula,
                    "confianza_matricula": confianza_matricula,
                    "mx1": mx1,
                    "my1": my1,
                    "mx2": mx2,
                    "my2": my2,
                    "texto_matricula": texto_matricula
                })

                #Se asigna un color diferente a cada clase de objeto
                if cls == 0:
                    R = 128
                    G = 0
                    B = 0
                elif cls == 1:
                    R = 0
                    G = 128
                    B = 0
                elif cls == 2:
                    R = 0
                    G = 0
                    B = 128

                #Se dibuja el rectángulo y la etiqueta del objeto detectado
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)

                cv2.putText(img, track_id + ' ' + nombre_clase, [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

                #Se muestra la información de la matrícula cuando se ha asociado a un coche
                if matricula_asociada is not None:

                    texto = texto_matricula

                    if texto == "":
                        texto = "Matricula no reconocida"

                    cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)

                    cv2.putText(img, texto, (mx1, max(20, my1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        #Se muestran los contadores de personas, bicicletas y coches en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        #Se guarda el fotograma procesado en el vídeo de salida
        out.write(img)

        #Se muestra el fotograma procesado
        cv2.imshow('Vid', img)

    else:
        break

    #Se comprueba si se ha pulsado ESC para detener el procesamiento
    if cv2.waitKey(20) == 27:
        break

#Se libera el objeto de captura del vídeo
vid.release()

#Se libera el objeto de escritura del vídeo
out.release()

#Se cierra el archivo CSV
archivo_csv.close()

#Se cierran las ventanas de visualización
cv2.destroyAllWindows()